In [3]:
print("Pene de Lur")
print("Nigga")
print("Hola Luristán")
print("iñigo therian")

Pene de Lur
Nigga
Hola Luristán
iñigo therian


In [10]:
### IMPORTS ###
import pandas as pd
from pydrive2.auth import GoogleAuth
from pydrive2.drive import GoogleDrive
from sklearn.preprocessing import StandardScaler, OrdinalEncoder, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split

DATA PROCESSING

In [12]:
### DATASET LOADING ###

gauth = GoogleAuth()
gauth.LocalWebserverAuth()
drive = GoogleDrive(gauth)

# Google Drive ID for public sharing of the dataset
file_id = "1aGGJNl1VtET_6HqQh81ouSKqe6-lN-OG"
file = drive.CreateFile({"id": file_id})
file.GetContentFile("ai_tools.csv")

# Reading the csv and loading it into a pandas dataframe.
df = pd.read_csv("ai_tools.csv")
df.head()

# The target is already a categorical multiclass column — no transformation needed.
# Classes: "Free", "Freemium", "Subscription", "Pay-as-you-go", "Open Source", "Usage-Based"

# We remove the target from the features to avoid data leakage.
X = df.drop(["Pricing_Model"], axis=1)

# We only keep the label.
y = df[["Pricing_Model"]]

print(X)
print(y)

InvalidConfigError: Invalid client secrets file ('Error opening file', 'client_secrets.json', 'No such file or directory', 2)

In [13]:
### MISSING DATA ANALYSIS ###

# Total missing values per column
print("Missing values per column:")
print(df.isnull().sum())

# Percentage of missing values per column
print("\nMissing values (%):")
print((df.isnull().sum() / len(df) * 100).round(2))

# Quick summary: any missing at all?
print(f"\nTotal missing values in dataset: {df.isnull().sum().sum()}")

Missing values per column:


NameError: name 'df' is not defined

In [ ]:
### TRANSFORMATION PIPELINE ###
# We drop non-informative columns
cols_to_drop = ['AI_Name', 'Developer', 'Website_URL', 'Key_Functionality']
X = df.drop(columns=cols_to_drop + ['Pricing_Model'])
y = df['Pricing_Model']

# Train / Validation / Test split (80-10-10)
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
)

print(f"Train size:      {X_train.shape[0]}")
print(f"Validation size: {X_val.shape[0]}")
print(f"Test size:       {X_test.shape[0]}")

# We define column groups for encoding
ordinal_cols = ['Context_Window']
ordinal_categories = [['N/A', '8k', '32k', '128k', '1M']]  # explicit order

onehot_cols = ['Intelligence_Type', 'Primary_Domain', 'API_Availability', 'Accessibility']

numerical_cols = ['Release_Year', 'Popularity_Votes']

# we build the ColumnTransformer
preprocessor = ColumnTransformer(transformers=[
    ('ordinal', OrdinalEncoder(categories=ordinal_categories), ordinal_cols),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False), onehot_cols),
    ('scaler', StandardScaler(), numerical_cols),
])

# we fit ONLY on train, then transform each split
X_train_processed = preprocessor.fit_transform(X_train)
X_val_processed   = preprocessor.transform(X_val)
X_test_processed  = preprocessor.transform(X_test)

# Optional: recover a readable DataFrame
onehot_feature_names = preprocessor.named_transformers_['onehot'].get_feature_names_out(onehot_cols)
feature_names = ordinal_cols + list(onehot_feature_names) + numerical_cols

X_train_processed = pd.DataFrame(X_train_processed, columns=feature_names)
X_val_processed   = pd.DataFrame(X_val_processed,   columns=feature_names)
X_test_processed  = pd.DataFrame(X_test_processed,  columns=feature_names)

print(X_train_processed.head())
print(f"\nFinal shape — Train: {X_train_processed.shape}, Val: {X_val_processed.shape}, Test: {X_test_processed.shape}")